# TA 1: linear spectra and wavelength calibration (ITOS)

Template on 250709: `s00` is the ITOS filter, `s01` the sample. Run it first: `ta_2_processing`
uses the calibration it writes.

Steps: reject bad repetitions and average each linear spectrum, calibrate the wavelength axis
on the ITOS spectrum, write the calibration file, save the spectra on the calibrated axis.
Raw files are never rewritten.

Each interactive tool is preceded by a **decisions** cell: paste there what its "Done" button
prints. The step after the tool uses that literal, not the widget, so "Run all" gives the same
result without clicking.

Needs `%matplotlib widget` (JupyterLab or VS Code, with `ipympl`).

In [ ]:
%matplotlib widget
import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from edyx.calib import interp_calibration, write_calibration
from edyx.tana import itos_calibration, load_itos_spectrum
from pyspan import Linear

## Parameters

In [ ]:
data_dir = Path("../../edyx/examples/data")  # edyx example data; in the lab: Path(".")
fnames = sorted(data_dir.glob("s0?.dat"))     # linear spectra, with repetitions
itos_name = "s00.dat"                         # the ITOS filter
peaks = dict(width=4, prominence=40, height=200)  # choice: ITOS peak picking, check the plot
date = datetime.date(2025, 7, 9)

out_dir = Path("output")
wl_file = out_dir / f"{date:%y%m%d}_wl_itos.toml"
[f.name for f in fnames]

## Reject bad repetitions and average

RMSD of each repetition against the median of its file. Default threshold: mean + 3.5 std of
the RMSD, applied to all files at once.

In [ ]:
# Paste the output of "Done" here. [] = default threshold
threshold = []

In [ ]:
lin = Linear.from_files(fnames, threshold=threshold)
lin.plot()

In [ ]:
wl_raw = lin.wl  # measured axis
spectra = {f.name: od for f, od in zip(fnames, lin.mean_spectra)}  # mean of the kept repetitions, mOD

## Wavelength calibration

The same peak picking on the measured and the reference ITOS spectra, peaks paired in order,
linear fit (`tana.itos_calibration`). Different peak counts raise an error: change `peaks`.
Equal counts do not guarantee the right pairs: check the plot and the residuals.

In [ ]:
cal = itos_calibration(wl_raw, spectra[itos_name], **peaks)
observed, assigned = cal["fit"]["points"].T
coef = cal["fit"]["coef"]  # wl = coef[0] + coef[1] * wl_raw
print("coef:", coef)
print("residuals / nm:", np.round(assigned - (coef[0] + coef[1] * observed), 2))

wl_ref, spec_ref = load_itos_spectrum()
fig, ax = plt.subplots()
ax.plot(wl_raw, spectra[itos_name], lw=0.8, label=f"{itos_name}, measured axis")
ax.plot(wl_ref, spec_ref, lw=0.8, label="reference")
ax.plot(observed, np.interp(observed, wl_raw, spectra[itos_name]), "+", color="C0")
ax.plot(assigned, np.interp(assigned, wl_ref, spec_ref), "x", color="C1")
ax.set_xlim(wl_raw.min(), wl_raw.max())
ax.set_xlabel("wavelength / nm")
ax.set_ylabel("A / mOD")
ax.legend()

In [ ]:
out_dir.mkdir(exist_ok=True)
write_calibration(wl_file, "wl", "wl_raw", **cal, units=["nm", "nm"], date=date,
                  source=itos_name, made_by="ta_1_linear_itos.ipynb")

## Save the spectra on the calibrated axis

The ITOS spectrum is saved on the measured axis: it is the input of the calibration.

In [ ]:
wl = interp_calibration(cal, wl_raw)
others = [name for name in spectra if name != itos_name]

np.savetxt(out_dir / f"{Path(itos_name).stem}_ITOS_ab_{date:%y%m%d}.dat",
           np.column_stack([wl_raw, spectra[itos_name]]), fmt="%.6e",
           header="measured wavelength / nm, A / mOD")
for name in others:
    np.savetxt(out_dir / f"{Path(name).stem}_ab_{date:%y%m%d}.dat",
               np.column_stack([wl, spectra[name]]), fmt="%.6e",
               header=f"wavelength / nm (calibration: {wl_file.name}), A / mOD")

fig, ax = plt.subplots()
for name in others:
    ax.plot(wl, spectra[name], label=name)
ax.set_xlabel("wavelength / nm")
ax.set_ylabel("A / mOD")
ax.legend()
fig.savefig(out_dir / f"linears_{date:%y%m%d}.png", dpi=300)

## Check against the lab's output (example only)

The lab's `s01_ab_250709.dat`: averaged by `Linear`, then rewritten onto the calibrated axis by
`ITOS_peak.ipynb`.

In [ ]:
wl_lab, od_lab = np.loadtxt(data_dir / "s01_ab_250709.dat").T
print("max |difference|, wl / nm:", np.max(np.abs(wl - wl_lab)),
      "; A / mOD:", np.max(np.abs(spectra["s01.dat"] - od_lab)))
assert np.allclose(wl, wl_lab, atol=1e-3) and np.allclose(spectra["s01.dat"], od_lab, atol=1e-3)